<a href="https://colab.research.google.com/github/anjalidrj/GO-analysis-of-peptides/blob/main/hit_count_from_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Extract hit gene sequences from a csv
Upload your Excel hits file (with a `transcript_id` / `transcript id` column) and your transcriptome FASTA. This pulls out just the sequences for genes that had hits, ready for downstream annotation (eggNOG-mapper, BLAST, etc.).

In [ ]:
# Cell 1: install dependencies
!pip install -q biopython pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 13.6 MB/s eta 0:00:00


In [ ]:
# Cell 2: upload your CSV file (the hits file)
from google.colab import files
print("Upload your CSV file (hits file)...")
uploaded_csv = files.upload()
CSV_FILE = list(uploaded_csv.keys())[0]
print('Uploaded:', CSV_FILE)

Upload your CSV file (hits file)...


Saving exact_hits_final_sol.csv to exact_hits_final_sol.csv
Uploaded: exact_hits_final_sol.csv


In [ ]:
# Cell 3: upload your transcriptome FASTA (the full set of gene/transcript sequences)
print("Upload your transcriptome FASTA file...")
uploaded_fasta = files.upload()
FASTA_FILE = list(uploaded_fasta.keys())[0]
print('Uploaded:', FASTA_FILE)

Upload your transcriptome FASTA file...


Saving sol.fasta to sol.fasta
Uploaded: sol.fasta


In [ ]:
# Cell 4: config -- adjust if your sheet/column names differ
TRANSCRIPT_COL = 'gene_id'   # the column in your Excel file with transcript IDs
OUTPUT_FASTA = 'hit_sequences.fasta'

# If your transcript IDs in Excel might have version suffixes (e.g. .1, .2) that don't
# always match the FASTA headers exactly, set this to True to match on the ID
# ignoring version numbers.
IGNORE_VERSION_SUFFIX = False

In [ ]:
# Cell 5: load the CSV file, get unique transcript IDs, and count hits per transcript
import pandas as pd
import re

df = pd.read_csv(CSV_FILE)
print(f"Loaded {len(df)} rows from {CSV_FILE}")
print(f"Columns: {list(df.columns)}")

if TRANSCRIPT_COL not in df.columns:
    raise ValueError(
        f"Column '{TRANSCRIPT_COL}' not found. Available columns: {list(df.columns)}\n"
        f"Update TRANSCRIPT_COL in Cell 4 to match."
    )

# count how many hit rows (peptide matches) each transcript has
hit_counts = (
    df[TRANSCRIPT_COL]
    .dropna()
    .astype(str)
    .value_counts()
    .rename_axis('transcript_id')
    .reset_index(name='total_transcript_hits')
    .sort_values('total_transcript_hits', ascending=False)
)

print(f"\nUnique transcript IDs to extract: {len(hit_counts)}")
print("\nTop 10 most-hit transcripts:")
print(hit_counts.head(10).to_string(index=False))

hit_ids = set(hit_counts['transcript_id'])
# ordered list, most-hit first -- used to control the order sequences get written in Cell 6
hit_ids_ordered = list(hit_counts['transcript_id'])

if IGNORE_VERSION_SUFFIX:
    hit_ids_stripped = {re.sub(r'\.\d+$', '', tid) for tid in hit_ids}
else:
    hit_ids_stripped = None

Loaded 6175 rows from exact_hits_final_sol.csv
Columns: ['peptide_id', 'peptide_seq', 'peptide_length', 'hydrophilicity', 'transcript_id', 'gene_description', 'frame', 'position_aa', 'total_transcript_hits', 'gene_id']

Unique transcript IDs to extract: 5103

Top 10 most-hit transcripts:
        transcript_id  total_transcript_hits
gene-Solyc07g061900.3                     20
gene-Solyc02g089260.4                      5
gene-Solyc08g006230.4                      5
gene-Solyc01g009180.4                      5
gene-Solyc08g079180.3                      5
gene-Solyc08g080690.4                      5
gene-Solyc12g100360.1                      4
gene-Solyc09g015705.1                      4
gene-Solyc01g108660.4                      4
gene-Solyc08g006420.3                      4


In [ ]:
# Cell 6: scan the FASTA, collect matching sequences, write them sorted by hit count
from Bio import SeqIO
from tqdm import tqdm
import re

gene_re = re.compile(r'gene:(\S+)')       # pulls GLYMA_.../Vitvi... from the header
sequences_by_id = {}                       # gene_id -> longest transcript sequence

for record in tqdm(SeqIO.parse(FASTA_FILE, 'fasta'), desc='Scanning FASTA'):
    m = gene_re.search(record.description)
    gene = m.group(1) if m else record.id  # fall back to record.id if no gene: field
    if gene in hit_ids:
        seq = str(record.seq)
        # keep the longest isoform as the representative for each gene
        if gene not in sequences_by_id or len(seq) > len(sequences_by_id[gene]):
            sequences_by_id[gene] = seq

# write in hit-count order (most-hit gene first)
written = 0
with open(OUTPUT_FASTA, 'w') as out:
    for tid in hit_ids_ordered:
        if tid in sequences_by_id:
            out.write(f'>{tid}\n{sequences_by_id[tid]}\n')
            written += 1

print(f"\nWrote {written} sequences to {OUTPUT_FASTA} (sorted by hit count, highest first)")
print(f"Requested {len(hit_ids)} unique gene IDs")

hit_counts.to_csv('hit_counts_summary.csv', index=False)
print("Saved hit_counts_summary.csv")

if written < len(hit_ids):
    missing = [t for t in hit_ids if t not in sequences_by_id]
    print(f"\n[NOTE] {len(missing)} gene IDs not found in the FASTA.")
    print(f"Sample missing: {missing[:5]}")

Scanning FASTA: 33562it [00:00, 70189.77it/s]


Wrote 5103 sequences to hit_sequences.fasta (sorted by hit count, highest first)
Requested 5103 unique gene IDs
Saved hit_counts_summary.csv


In [ ]:
# Cell 7: download the result
files.download(OUTPUT_FASTA)
files.download('hit_counts_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>